#### **1. The Big Picture: How Biopython Organizes Sequence Data**
---

When working with biological sequences, there are three conceptual levels in Biopython:

1️⃣	Seq() : The raw biological sequence (DNA, RNA, or protein).	The actual text of the gene/protein.

2️⃣	SeqRecord(): A record that wraps a Seq object and adds metadata — identifiers, descriptions, features, annotations, etc.	A page in GenBank or EMBL describing that sequence.

3️⃣	SeqFeature() : Describes specific regions or features on that sequence (e.g., genes, CDS, promoters).	Labels or highlights marking interesting parts on the sequence map.

In [1]:
# importing libraries
import pandas as pd
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord
from Bio.SeqFeature import SeqFeature
from datetime import datetime

#### **1.1 Populating the SeqRecord befroe instantiating the SeqRecord Object**
---

In [22]:
# Let's start with an example of a SeqRecord
seq1 = Seq('ATGCGTACGTAG') # Seq object
Record1 = SeqRecord(
    seq= seq1,
    id = '1493', # adding an identifier
    name = 'AB2A_toxin',
    description= 'Salmonella virulence factor',
    annotations={'Date': datetime(2025,10,29)}, # adding a datetime of the creation day
    features =[]
) 

In [23]:
# let's now access some attributes:seq, id, name, descriptions etc
Record1.seq     # shows the sequence

Seq('ATGCGTACGTAG')

In [24]:
# let's print the ID and name
print('ID :', Record1.id)
print('Name :',Record1.name)
print(Record1.features)

ID : 1493
Name : AB2A_toxin
[]


In [7]:
# Let's print the description and annotion
print('Description :',Record1.description)
print("----" * 10)
print('Annotation:',Record1.annotations['Date']) # select the specific annotation or you can just get the whole dict

Description : Salmonella virulence factor
----------------------------------------
Annotation: 2025-10-29 00:00:00


#### **1.2 You can equally Create the SeqRecord Object and Later Populate it**
----

In [8]:
# SeqRecord will need at least the Seq object to instantiate an boject
Rec2 = SeqRecord(seq1)

# let's add the id, name, description and etc
Rec2.id = '1493'
Rec2.name = 'AB2A_toxin'
Rec2.description = 'Salmonella typhi virulence factor'
Rec2.annotations = {'Date': datetime(2025,10,29)}

In [9]:
# let's see if it worked
print('ID :', Rec2.id)
print('Name :',Rec2.name)
print('Annotation :', Rec2.annotations['Date'])
# it worked

ID : 1493
Name : AB2A_toxin
Annotation : 2025-10-29 00:00:00


In [11]:
# Let's read one fasta file from ncbi and check this information
from Bio import SeqIO
record_3 = SeqIO.read("single_record.fasta",'fasta') 

In [18]:
# let's check the sequence , id , name and descriptions
len(record_3.seq)

151

In [ ]:
# Descriptions 
print(record_3.description)
#Id, name
print('ID :', record_3.id)
print('Name :', record_3.name)

demo_seq_001 Demo DNA sequence for SeqIO.read
ID : demo_seq_001
Name : demo_seq_001


#### **1.3 SeqFeature: The Core Idea**
---
A SeqFeature answers three main questions about a part of a sequence:

| Question                          | Attribute     | Example                                        |
| --------------------------------- | ------------- | ---------------------------------------------- |
| **What type of feature is this?** | `.type`       | `"gene"`, `"CDS"`, `"promoter"`                |
| **Where is it located?**          | `.location`   | `SimpleLocation(100, 900, strand=1)`           |
| **What details describe it?**     | `.qualifiers` | `{"gene": ["ABC1"], "note": ["hypothetical"]}` |


In [20]:
# Let's work with some SeqFeatures object
from Bio.SeqFeature import SeqFeature, SimpleLocation

In [30]:
feature = SeqFeature(
    location= SimpleLocation(start= 10, end= 25, strand= None),
    type= 'gene',
    qualifiers={'gene':['CB21'], 'Note':['Not Real: Just made it up']}
    )

In [26]:
# let's append this to the feature list in Record3
record_3.features.append(feature) # features is a list, so we append the record object features with our feature

In [29]:
# Let's check if the record is now having the feature 
record_3.features

[SeqFeature(SimpleLocation(ExactPosition(10), ExactPosition(25)), type='gene', qualifiers=...)]

In [31]:
# let's check the positions and other stuff in the Seqfeature of this record
print('Start position:', feature.location.start)
print('Stop position:', feature.location.end)
print('Qualifiers:', feature.qualifiers['gene']) # selects the value of the gene in the qualifiers dict

Start position: 10
Stop position: 25
Qualifiers: ['CB21']


#### **1.4 The location Object: Defining the Region**
---

Every SeqFeature has a .location attribute that defines where on the parent sequence the feature lies.

A location combines:

- Start position

- End position

- Strand information

#### **a. Simple location**
---

In [32]:
# Example of a location
loc = SimpleLocation(100, 900, strand=1)
loc.start

ExactPosition(100)

In [33]:
loc.end

ExactPosition(900)

#### **1.5 Compound and Fuzzy Locations**
---
**(b) CompoundLocation**

Sometimes a gene or CDS spans multiple regions (e.g., exons).

In [34]:
# let's import the compoundlocation function
from Bio.SeqFeature import CompoundLocation

In [36]:
# we will create simple locations for each of the location for our features
loc1 = SimpleLocation(start= 100, end= 200)
loc2 = SimpleLocation(300,400)

# Now we pass these simple location to a compoundlocation object in our SeqFeautrue objec
comp_loc = CompoundLocation([loc1,loc2])

# Now we can pass it to a seqfeature object
comp_feature = SeqFeature(comp_loc, type = 'CDS',qualifiers= {'Loc1':'Staph Bleomycin1','Loc2':'Antimicrobial Resis'})

In [37]:
# Now let's try to add this feature to our record 3 object
record_3.features.append(comp_feature)

In [38]:
# since we have two feautures, one is the simple location and the other is compound location
for feature in record_3.features:
    print(f'Type :{feature.type}\n location : {feature.location}')

Type :gene
 location : [10:25]
Type :CDS
 location : join{[100:200], [300:400]}



**(c) Fuzzy Positions**

Biology is rarely exact- transcription start sites, cleavage sites, etc., can be uncertain.
We may know it starts after,before, etc of some point but not the exact positions, that is when fuzzy positions come in and,
Biopython provides classes like:

| Class            | Meaning                  | Example    |
| ---------------- | ------------------------ | ---------- |
| `ExactPosition`  | exact                    | `100`      |
| `BeforePosition` | before a point           | `<100`     |
| `AfterPosition`  | after a point            | `>100`     |
| `WithinPosition` | between two points       | `(8^9)`    |
| `OneOfPosition`  | multiple possible points | `{45, 50}` |


In [39]:
# Let's import the locations classes 
from Bio.SeqFeature import AfterPosition, BeforePosition,BetweenPosition,OneOfPosition

In [41]:
# Let's add a fuzzy location object
startf = BeforePosition(5)
endf = BetweenPosition(position= 9, left= 8 , right= 9) # between position 8 and 9 and the right is the anchor position.
fuzzy_loc = SimpleLocation(start= startf, end = endf)

In [42]:
# Let's look at our output our fuzzy_location
print(fuzzy_loc)

[<5:(8^9)]


In [43]:
# Now we have our fuzzy location object and we can put into a SeqFeature object
# let's check some parts of our location
print('Start:', fuzzy_loc.start)
print('End:', fuzzy_loc.end)

Start: <5
End: (8^9)


In [44]:
# Let's work on another fuzzy location object :
start = BetweenPosition(position= 3, left= 2, right= 3)
end = BeforePosition(10)
# let's now create a simple location with our fuzzy start and end 
fuzy_loc2 = SimpleLocation(start, end, strand = -1)

In [45]:
# let's see our location details 
print('Start:', fuzy_loc2.start)
print('End:', fuzy_loc2.end)

Start: (2^3)
End: <10


#### **1.6 Extracting Sequences from Features**
---

A SeqFeature doesn’t contain the sequence, but it knows how to find it on its parent sequence.

In [49]:
# let's create our sequence
seq1 = Seq("ACCGAGACGGCAAAGGCTAGCATAGGTATGAGACTTCCTTCCTGCCAGTGCTGAGGA")
location3 = SimpleLocation(start=3, end= 11)

# let's use the location object to slice our sequence
location3.extract(seq1)

Seq('GAGACGGC')

In [51]:
# let's create a feature object with our location 
feature_com = SeqFeature(location=location3, type="CDS",qualifiers= {'Gene_info':'Staph aureus bleomycin'})
# let's create a SeqRecord object and add the sequence and feature to it
Record_com = SeqRecord(
    seq= seq1,  # Sequence object added
    id = "Staph113",
    name = 'Bleomycin',
    description= 'This is a gene that confers antimicrobial resistance',
    features= [feature_com]  # feature object added
)

In [52]:
for feature in Record_com.features:
    print(feature.location)
    print(feature.type)

[3:11]
CDS


In [55]:
# Let's use a genbank Data:
gene_record = SeqIO.read(r"D:\BIOINFORMATICS\Gene_files\sequence (1).gb",'genbank')

# Let's look throught the first ten features 
features_10 =gene_record.features[0:10]
features_10

[SeqFeature(SimpleLocation(ExactPosition(0), ExactPosition(48502), strand=1), type='source', qualifiers=...),
 SeqFeature(SimpleLocation(ExactPosition(190), ExactPosition(736), strand=1), type='gene', qualifiers=...),
 SeqFeature(SimpleLocation(ExactPosition(190), ExactPosition(736), strand=1), type='CDS', qualifiers=...),
 SeqFeature(SimpleLocation(ExactPosition(710), ExactPosition(2636), strand=1), type='gene', qualifiers=...),
 SeqFeature(SimpleLocation(ExactPosition(710), ExactPosition(2636), strand=1), type='CDS', qualifiers=...),
 SeqFeature(SimpleLocation(ExactPosition(2632), ExactPosition(2839), strand=1), type='gene', qualifiers=...),
 SeqFeature(SimpleLocation(ExactPosition(2632), ExactPosition(2839), strand=1), type='CDS', qualifiers=...),
 SeqFeature(SimpleLocation(ExactPosition(2835), ExactPosition(4437), strand=1), type='gene', qualifiers=...),
 SeqFeature(SimpleLocation(ExactPosition(2835), ExactPosition(4437), strand=1), type='CDS', qualifiers=...),
 SeqFeature(SimpleLo

In [56]:
# Let's check grab the feature at index 3
feature_3 = features_10[3]
feature_3

SeqFeature(SimpleLocation(ExactPosition(710), ExactPosition(2636), strand=1), type='gene', qualifiers=...)

In [58]:
# Qualifiers of the features
feature_3.qualifiers

{'gene': ['A'], 'locus_tag': ['lambdap02'], 'db_xref': ['GeneID:2703524']}

In [62]:
# using the feature to extract the sequence
seq_fea_3 = feature_3.extract(gene_record.seq)
seq_fea_3

Seq('GTGAATATATCGAACAGTCAGGTTAACAGGCTGCGGCATTTTGTCCGCGCCGGG...TGA')

In [63]:
# Checking the length of the extracted sequence
len(seq_fea_3)

1926

#### **1.7 Location Checking**
---
- You can use the Python keyword "in" with a SeqFeature or location object to see if the base/residue for a parent coordinate is within the feature/location or not.

In [67]:
# Let's say we are checking for this feature with as start location of 2632
interest = 2632
for feature in gene_record.features:
    if interest in feature:
        print(f'{feature.type} : {feature.location}: {feature.qualifiers}')

source : [0:48502](+): {'organism': ['Escherichia phage Lambda'], 'mol_type': ['genomic DNA'], 'host': ['Escherichia coli'], 'db_xref': ['taxon:2681611']}
gene : [710:2636](+): {'gene': ['A'], 'locus_tag': ['lambdap02'], 'db_xref': ['GeneID:2703524']}
CDS : [710:2636](+): {'gene': ['A'], 'locus_tag': ['lambdap02'], 'codon_start': ['1'], 'transl_table': ['11'], 'product': ['terminase large subunit'], 'protein_id': ['NP_040581.1'], 'db_xref': ['GeneID:2703524'], 'translation': ['MNISNSQVNRLRHFVRAGLRSLFRPEPQTAVEWADANYYLPKESAYQEGRWETLPFQRAIMNAMGSDYIREVNVVKSARVGYSKMLLGVYAYFIEHKQRNTLIWLPTDGDAENFMKTHVEPTIRDIPSLLALAPWYGKKHRDNTLTMKRFTNGRGFWCLGGKAAKNYREKSVDVAGYDELAAFDDDIEQEGSPTFLGDKRIEGSVWPKSIRGSTPKVRGTCQIERAASESPHFMRFHVACPHCGEEQYLKFGDKETPFGLKWTPDDPSSVFYLCEHNACVIRQQELDFTDARYICEKTGIWTRDGILWFSSSGEEIEPPDSVTFHIWTAYSPFTTWVQIVKDWMKTKGDTGKRKTFVNTTLGETWEAKIGERPDAEVMAERKEHYSAPVPDRVAYLTAGIDSQLDRYEMRVWGWGPGEESWLIDRQIIMGRHDDEQTLLRVDEAINKTYTRRNGAEMSISRICWDTGGIDPTIVYERSKKHGLFRVIPIKGASVYGKPVASMPRKRNKNGVYLTEIGT

In [68]:
# let's make the display a little readable:
interest = 2632
for feature in gene_record.features:
    if interest in feature:
        gene_name = feature.qualifiers.get('gene', [''])[0]
        product = feature.qualifiers.get('product', [''])[0]
        print(f"{feature.type.upper():6} | Gene: {gene_name:5} | Product: {product}")


SOURCE | Gene:       | Product: 
GENE   | Gene: A     | Product: 
CDS    | Gene: A     | Product: terminase large subunit
GENE   | Gene: W     | Product: 
CDS    | Gene: W     | Product: head-tail adaptor Ad1


#### **1.8 Sequence described by a feature or location**
---
- A SeqFeature or location object doesn’t directly contain a sequence, instead the location (see Section Positions and locations) describes how to get this from the parent sequence. 
- For example consider a (short) gene sequence with location 5:18 on the reverse strand, which in GenBank/EMBL notation using 1-based counting would be complement(6..18), like this:

In [69]:
# Let's take an example 
seqF = Seq("ACCGAGACGGCAAAGGCTAGCATAGGTATGAGACTTCCTTCCTGCCAGTGCTGAGGAACTGGGAGCCTAC")
featureF = SeqFeature(SimpleLocation(5, 18, strand= +1), type="gene")

In [70]:
# let's slice the sequece with the feature location
seqF[featureF.location.start: featureF.location.end]

Seq('GACGGCAAAGGCT')

In [71]:
# we could have equally used the feature extract method to get the same sequence
featureF.extract(seqF)

Seq('GACGGCAAAGGCT')

In [72]:
# Let's change the location to read the template strnad or antisense strand 
featureR = SeqFeature(SimpleLocation(5, 18, strand= -1), type="gene")
featureR.extract(seqF)
# You see it's just the reverse complement of the coding strand

Seq('AGCCTTTGCCGTC')

##### **Note that the length of a feature is the length of the sequence it location shows**

In [43]:
# let's look at this 
len(featureR)
# you see that it is 13

13

#### **1.9 Comparison of SeqRecord Object**
---
- The SeqRecord objects can be very complex, but here’s a simple example
- Older versions of biopython would have used python's comparison operator to return True or False
- New version of biopython appreciates these complexity and recommends comparison of Record arguments instead
- Example is to compare the id, the Seq, the name, features etc

In [73]:
# Let's look at these examples
record1 = SeqRecord(Seq("ACGT"), id="test")
record2 = SeqRecord(Seq("ACGT"), id="test")

In [74]:
# let's compare ids across the two records 
print(record1.id == record2.id)
# comparing the sequence in the two records
print(record1.seq == record2.seq)

True
True


In [75]:
# if we tried to compare the records itself, it wouldn't work
#record1 == record2 : says SeqRecord comparison is deliberately not implemented

In [76]:
# Now let's add a simple location object and try to compare : I just made this up 
loc1 = SeqFeature(location= SimpleLocation(start=0, end=2, strand = +1),type="Promoter", qualifiers={'Gene':['ABC']})
loc2 = SeqFeature(location= SimpleLocation(start=1, end=3, strand = -1),type="Promoter", qualifiers={'Gene':['CBA']})

In [77]:
# let's add these locations to our two SeqRecord objects
record1.features = loc1
record2.features = loc2

In [78]:
# Checking if the features in these two records starts at the same position
record1.features.location.start == record2.features.location.start

False

In [79]:
# let's check the end locations 
record1.features.location.end == record2.features.location.end

False

#### **2.0 : References**
----
- Another common annotation related to a sequence is a reference to a journal or other published work dealing with the sequence. We have a fairly simple way of representing a Reference in Biopython 
- we have a Bio.SeqFeature.Reference class that stores the relevant information about a reference as attributes of an object.
- The attributes include things that you would expect to see in a reference like journal, title and authors. 
- Additionally, it also can hold the medline_id and pubmed_id and a comment about the reference. These are all accessed simply as attributes of the object.

In [81]:
# importing the reference class
from Bio.SeqFeature import Reference, FeatureLocation

In [82]:
# initializing a reference object
ref = Reference()

# adding title, author, journal, pubmedid, locationin of the reference sequence
ref.title = 'Genetic analysis of sickle hemoglobin gene'
ref.authors = 'Samuel Abugare: Jeoming Song'
ref.journal = "Journal of Pathway Studies (2025) 17:99-108"
ref.pubmed_id = "543784"
ref.location = FeatureLocation(start=1,end=3) 

In [83]:
# Note that the reference is part of the annotation, which is a dictionary. 
# What that means is that we have to create a reference key and then the value will be this reference item 
# let's add this reference to the record object 1
record1.annotations['references'] = [ref]

# Note: You could only append if the key 'references' already exist in the annotation.

In [84]:
# Let's loop through the annotation reference list

for ref in record1.annotations['references']:
    print('Title  | : ', ref.title)
    print('Authors| :', ref.authors)
    print('Journal|:', ref.journal)

Title  | :  Genetic analysis of sickle hemoglobin gene
Authors| : Samuel Abugare: Jeoming Song
Journal|: Journal of Pathway Studies (2025) 17:99-108


In [85]:
# Let's say the referece key already existed within the record as it is in record1 now:
# We could just call the key, reference, and append it values that is the instantiated reference object
# Let's write a function that makes references 
def Ref_Maker(authors,title,journal,start,end ,pubmed_id):
    ref = Reference()
    ref.authors = authors
    ref.title = title
    ref.journal = journal
    ref.location = FeatureLocation(start= start, end = end)
    ref.pubmed_id = pubmed_id
    return ref

In [86]:
# let's make a reference object with our function
ref2 = Ref_Maker('Samubug;Jeoming Song','CRISPR Gene Editing','Pathway Studies',0,2,8435)

In [87]:
# let's now append this sencond refence to the record1 object annotations
record1.annotations['references'].append(ref2)
# see that it works

In [88]:
# let's loop through the references again to check if the second reference was added
for ref in record1.annotations['references']:
    print('Title  | : ', ref.title)
    print('Authors| :', ref.authors)
    print('Journal|:', ref.journal)
    print('***************************************************')

Title  | :  Genetic analysis of sickle hemoglobin gene
Authors| : Samuel Abugare: Jeoming Song
Journal|: Journal of Pathway Studies (2025) 17:99-108
***************************************************
Title  | :  CRISPR Gene Editing
Authors| : Samubug;Jeoming Song
Journal|: Pathway Studies
***************************************************


#### **2.1 The format method**
---
- SeqRecord.format() is basically a shortcut for saying “give me this record as if I was writing it out using SeqIO”. 
- You pick a format string (like "fasta", "genbank", "embl", etc.), and it returns a string in that file format.

In [59]:
# Let's create a seqRecord to demonstrate this
record = SeqRecord(
    Seq(
        "MMYQQGCFAGGTVLRLAKDLAENNRGARVLVVCSEITAVTFRGPSETHLDSMVGQALFGD"
        "GAGAVIVGSDPDLSVERPLYELVWTGATLLPDSEGAIDGHLREVGLTFHLLKDVPGLISK"
        "NIEKSLKEAFTPLGISDWNSTFWIAHPGGPAILDQVEAKLGLKEEKMRATREVLSEYGNM"
        "SSAC"
    ),
    id="gi|14150838",
    description="chalcone synthase [Cucumis sativus]",
    annotations= {'molecule_type':'protein'}
)

In [60]:
# let's say we wan it in a fasta format first
print(record.format('fasta'))

>gi|14150838 chalcone synthase [Cucumis sativus]
MMYQQGCFAGGTVLRLAKDLAENNRGARVLVVCSEITAVTFRGPSETHLDSMVGQALFGD
GAGAVIVGSDPDLSVERPLYELVWTGATLLPDSEGAIDGHLREVGLTFHLLKDVPGLISK
NIEKSLKEAFTPLGISDWNSTFWIAHPGGPAILDQVEAKLGLKEEKMRATREVLSEYGNM
SSAC



In [61]:
print(record.format('genbank'))

LOCUS       gi|14150838              184 aa                     UNK 01-JAN-1980
DEFINITION  chalcone synthase [Cucumis sativus].
ACCESSION   gi|14150838
VERSION     gi|14150838
KEYWORDS    .
SOURCE      .
  ORGANISM  .
            .
FEATURES             Location/Qualifiers
ORIGIN
        1 mmyqqgcfag gtvlrlakdl aennrgarvl vvcseitavt frgpsethld smvgqalfgd
       61 gagavivgsd pdlsverply elvwtgatll pdsegaidgh lrevgltfhl lkdvpglisk
      121 niekslkeaf tplgisdwns tfwiahpggp aildqveakl glkeekmrat revlseygnm
      181 ssac
//



##### **NB: that this only formats the record but doesn't save it or write it to a new file**

In [62]:
# let's create an ouput file and write this fasta/genbank format into it
with open('output.fasta','w') as fashandle:
    fashandle.write(record.format('fasta'))

In [63]:
# You see how that works
print(SeqIO.read('output.fasta','fasta'))

ID: gi|14150838
Name: gi|14150838
Description: gi|14150838 chalcone synthase [Cucumis sativus]
Number of features: 0
Seq('MMYQQGCFAGGTVLRLAKDLAENNRGARVLVVCSEITAVTFRGPSETHLDSMVG...SAC')


#### **2.2 Slicing a SeqRecord Object**
---
- You can slice a SeqRecord, to give you a new SeqRecord covering just part of the sequence. 
- What is important here is that any per-letter annotations are also sliced, and any features which fall completely within the new sequence are preserved (with their locations adjusted).

In [64]:
#let's use this data from ncbi genebank
yersinia =  SeqIO.read(r"C:\Users\Sam\Downloads\sequence (1).gb", "genbank")

# checking record object
print('num_seq:', len(yersinia.seq), 'bases')
print('num_fea:', len(yersinia.features), 'features')

num_seq: 9609 bases
num_fea: 21 features


In [65]:
# Randomly looking at a feature
print((yersinia.features[20]))

type: CDS
location: [8087:8429](-)
qualifiers:
    Key: codon_start, Value: ['1']
    Key: inference, Value: ['COORDINATES: similar to AA sequence:RefSeq:WP_002225538.1']
    Key: locus_tag, Value: ['YP_RS22250']
    Key: note, Value: ['Derived by automated computational analysis using gene prediction method: Protein Homology.']
    Key: old_locus_tag, Value: ['pPCP10', 'YP_pPCP10']
    Key: product, Value: ['type II toxin-antitoxin system RelE/ParE family toxin']
    Key: protein_id, Value: ['WP_002225538.1']
    Key: transl_table, Value: ['11']
    Key: translation, Value: ['MVLFSQRFDDWLNEQEDALQEKVLADLKKLQVYGPELPRPYADTVKGSRYKNMKELRVQFSGRPIRAFYAFDPIRRAIVLCAGDKSNDKRFYEKLVRIAEDEFTAHLNTLESK']



In [66]:
# Let’s slice this parent record from 4300 to 4800 (enough to include the pim gene/CDS)
sub_record = yersinia[4300:4800]

In [67]:
# let's check the length and the number of features we have in this sequence
print('Lenght:', len(sub_record.seq))
print('Num_Features:', len(sub_record.features))

Lenght: 500
Num_Features: 2


In [68]:
# le't now check what these two features are
print(sub_record.features[0])

type: gene
location: [84:480](+)
qualifiers:
    Key: locus_tag, Value: ['YP_RS22225']
    Key: old_locus_tag, Value: ['pPCP05', 'YP_pPCP05']



In [69]:
print(sub_record.features[1])

type: CDS
location: [84:480](+)
qualifiers:
    Key: codon_start, Value: ['1']
    Key: inference, Value: ['COORDINATES: similar to AA sequence:RefSeq:WP_001247799.1']
    Key: locus_tag, Value: ['YP_RS22225']
    Key: note, Value: ['Derived by automated computational analysis using gene prediction method: Protein Homology.']
    Key: old_locus_tag, Value: ['pPCP05', 'YP_pPCP05']
    Key: product, Value: ['pesticin immunity protein']
    Key: protein_id, Value: ['WP_223661135.1']
    Key: transl_table, Value: ['11']
    Key: translation, Value: ['MFLSSSGLAEKNTYTAKDILQNLELNTFGNSLSHGIYGKQTTFKQTEFTNIKSNTKKHIALINKDNSWMISLKILGIKRDEYTVCFEDFSLIRPPTYVAIHPLLIKKVKSGNFIVVKEIKKSIPGCTVYYH']



### **2.3 Adding SeqRecord Objects**
---
- You can add SeqRecord objects together, giving a new SeqRecord. 
- What is important here is that any common per-letter annotations are also added, all the features are preserved (with their locations adjusted), and any other common annotation is also kept (like the id, name and description).



In [73]:
# let's create a SeqRecord Object
Seq1 =  Seq('CCCTTCTTGTCTTCAGCGTTTCTCC')
Record = SeqRecord(
    seq= Seq1,
    id = '14223',
    description= 'Example Gene',
    letter_annotations= {'phred_quality':[26, 26, 18, 26, 26, 26, 26, 26,
                                           26, 26, 26, 26, 26, 26, 26, 22,
                                             26, 26, 26, 26, 26, 26, 26, 23, 23]})

In [76]:
# Now let's try and split this record into and then add it back together 
# Let's say we don't want the 20th T, that is we want ...TT.. and not ...TTT...
left = Record[:20] # this slices the record up to the 19th base
right = Record[21:] # jumps over 20, that is the T we don't want and slices the next bases that follows

In [79]:
# let's look at what happens to each of the annotations 
print(left.letter_annotations['phred_quality'])
print("**********************************")
print(right.letter_annotations['phred_quality'])

[26, 26, 18, 26, 26, 26, 26, 26, 26, 26, 26, 26, 26, 26, 26, 22, 26, 26, 26, 26]
**********************************
[26, 26, 23, 23]


In [80]:
# Now let's add the left and right record objects to create back the original record
combined = left + right 

In [82]:
print(combined.seq)
print(combined.letter_annotations['phred_quality'])

CCCTTCTTGTCTTCAGCGTTCTCC
[26, 26, 18, 26, 26, 26, 26, 26, 26, 26, 26, 26, 26, 26, 26, 22, 26, 26, 26, 26, 26, 26, 23, 23]


In [88]:
# this combination could have been equally achieved with this code :
combined2 = Record[:20] + Record[21:]
print(combined2.seq)
print(combined2.letter_annotations['phred_quality'])

CCCTTCTTGTCTTCAGCGTTCTCC
[26, 26, 18, 26, 26, 26, 26, 26, 26, 26, 26, 26, 26, 26, 26, 22, 26, 26, 26, 26, 26, 26, 23, 23]
